In [ ]:
# ----------------------------------------------------
# Cell 1: Install & Build Organ Donor Web Application
# ----------------------------------------------------
!pip install -q gradio pandas sqlite3 matplotlib

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from datetime import datetime

# ====================================================
# 1. DATABASE & COMPATIBILITY ALGORITHM CORE
# ====================================================
DB_NAME = "organ_donation_web.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS donors (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT, age INT, gender TEXT, blood TEXT, organ TEXT, location TEXT, hospital TEXT, contact TEXT
        )
    ''')
    c.execute('''
        CREATE TABLE IF NOT EXISTS recipients (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT, age INT, gender TEXT, blood TEXT, organ TEXT, urgency TEXT, location TEXT, hospital TEXT, contact TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

BLOOD_COMPATIBILITY = {
    "O-":  ["O-", "O+", "A-", "A+", "B-", "B+", "AB-", "AB+"],
    "O+":  ["O+", "A+", "B+", "AB+"],
    "A-":  ["A-", "A+", "AB-", "AB+"],
    "A+":  ["A+", "AB+"],
    "B-":  ["B-", "B+", "AB-", "AB+"],
    "B+":  ["B+", "AB+"],
    "AB-": ["AB-", "AB+"],
    "AB+": ["AB+"]
}

# Add initial sample data if database is empty
def add_sample_data():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT COUNT(*) FROM donors")
    if c.fetchone()[0] == 0:
        c.execute("INSERT INTO donors (name, age, gender, blood, organ, location, hospital, contact) VALUES ('David Miller', 34, 'Male', 'O-', 'Kidney', 'Bangalore', 'City Care Hospital', '9876543210')")
        c.execute("INSERT INTO donors (name, age, gender, blood, organ, location, hospital, contact) VALUES ('Sophia Taylor', 29, 'Female', 'A+', 'Kidney', 'Bangalore', 'Apollo Hospital', '9876543211')")
        c.execute("INSERT INTO donors (name, age, gender, blood, organ, location, hospital, contact) VALUES ('Robert Chen', 45, 'Male', 'B+', 'Heart', 'Delhi', 'AIIMS Hospital', '9876543212')")

        c.execute("INSERT INTO recipients (name, age, gender, blood, organ, urgency, location, hospital, contact) VALUES ('Alice Smith', 32, 'Female', 'A+', 'Kidney', 'Critical (Level 1)', 'Bangalore', 'Fortis Hospital', '9123456789')")
        c.execute("INSERT INTO recipients (name, age, gender, blood, organ, urgency, location, hospital, contact) VALUES ('Michael Brown', 50, 'Male', 'B+', 'Heart', 'High (Level 2)', 'Delhi', 'Max Hospital', '9123456788')")
        conn.commit()
    conn.close()

add_sample_data()

# ====================================================
# 2. WEB BACKEND FUNCTIONS
# ====================================================
def register_donor(name, age, gender, blood, organ, location, hospital, contact):
    if not name or not age or not location or not hospital or not contact:
        return "⚠️ Error: Please fill in all fields!"
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("INSERT INTO donors (name, age, gender, blood, organ, location, hospital, contact) VALUES (?,?,?,?,?,?,?,?)",
              (name, int(age), gender, blood, organ, location, hospital, contact))
    conn.commit()
    conn.close()
    return f"🎉 Donor '{name}' registered successfully for {organ} donation ({blood})!"

def register_recipient(name, age, gender, blood, organ, urgency, location, hospital, contact):
    if not name or not age or not location or not hospital or not contact:
        return "⚠️ Error: Please fill in all fields!"
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("INSERT INTO recipients (name, age, gender, blood, organ, urgency, location, hospital, contact) VALUES (?,?,?,?,?,?,?,?,?)",
              (name, int(age), gender, blood, organ, urgency, location, hospital, contact))
    conn.commit()
    conn.close()
    return f"🎉 Patient '{name}' registered successfully for {organ} transplant!"

def run_organ_matching(target_organ):
    conn = sqlite3.connect(DB_NAME)
    donors = pd.read_sql_query("SELECT * FROM donors WHERE organ=?", conn, params=[target_organ])
    recipients = pd.read_sql_query("SELECT * FROM recipients WHERE organ=?", conn, params=[target_organ])
    conn.close()

    if donors.empty or recipients.empty:
        return pd.DataFrame(), None, "⚠️ No compatible matches found for this organ."

    matches = []
    for _, r in recipients.iterrows():
        for _, d in donors.iterrows():
            allowed = BLOOD_COMPATIBILITY.get(d['blood'], [])
            if r['blood'] not in allowed:
                continue

            score = 50
            if d['blood'] == r['blood']: score += 20
            if "Critical" in r['urgency']: score += 20
            elif "High" in r['urgency']: score += 15
            elif "Medium" in r['urgency']: score += 10

            if d['location'].strip().lower() == r['location'].strip().lower(): score += 10
            if abs(d['age'] - r['age']) <= 10: score += 10

            matches.append({
                "Rank": "",
                "Match Score": f"{score}%",
                "Patient Name": r['name'],
                "Patient Blood": r['blood'],
                "Urgency": r['urgency'],
                "Donor Name": d['name'],
                "Donor Blood": d['blood'],
                "Location Match": "Same City" if d['location'].strip().lower() == r['location'].strip().lower() else "Different City",
                "Hospital": r['hospital'],
                "Score_Num": score
            })

    df = pd.DataFrame(matches)
    if df.empty:
        return pd.DataFrame(), None, "❌ No blood-compatible matches found."

    df = df.sort_values(by="Score_Num", ascending=False).reset_index(drop=True)
    df['Rank'] = [f"#{i+1}" for i in range(len(df))]
    df_display = df.drop(columns=["Score_Num"])

    # Generate Matplotlib Compatibility Graph
    plt.figure(figsize=(7, 3.5))
    colors = ['#2a9d8f' if s >= 80 else '#e76f51' for s in df['Score_Num']]
    names = [f"{r['Patient Name']}\n(vs {r['Donor Name']})" for _, r in df.iterrows()]
    plt.bar(names, df['Score_Num'], color=colors, width=0.4)
    plt.axhline(80, color='gray', linestyle='--', label='Optimal Match (80%)')
    plt.title(f"Organ Compatibility Score (%) - {target_organ}", fontweight='bold')
    plt.ylabel("Score (%)")
    plt.ylim(0, 100)
    plt.legend()
    plt.grid(axis='y', linestyle=':', alpha=0.6)
    plt.tight_layout()
    chart_path = "match_chart.png"
    plt.savefig(chart_path)
    plt.close()

    return df_display, chart_path, f"✅ Found {len(df_display)} compatible match pairing(s) for {target_organ}!"

def view_database():
    conn = sqlite3.connect(DB_NAME)
    donors = pd.read_sql_query("SELECT id, name, age, gender, blood, organ, location, hospital, contact FROM donors", conn)
    recipients = pd.read_sql_query("SELECT id, name, age, gender, blood, organ, urgency, location, hospital, contact FROM recipients", conn)
    conn.close()
    return donors, recipients

# ====================================================
# 3. GRADIO WEB USER INTERFACE (TEAL & BLUE MEDICAL THEME)
# ====================================================
theme = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="cyan",
    neutral_hue="slate"
)

with gr.Blocks(theme=theme, title="Organ Donor & Receiver Website") as app:
    gr.Markdown(
        """
        # 🫀 Organ Donor & Receiver Matching System
        ### *AI & Algorithmic Organ Compatibility Platform | NVIDIA Internship Project*
        """
    )

    with gr.Tabs():
        # TAB 1: MATCHING ENGINE
        with gr.TabItem("🧩 Smart Organ Matcher"):
            gr.Markdown("### Select organ type to calculate compatible donor-recipient pairings:")
            with gr.Row():
                organ_select = gr.Dropdown(choices=["Kidney", "Liver", "Heart", "Lungs", "Pancreas", "Cornea"], value="Kidney", label="Target Organ")
                btn_match = gr.Button("🔍 Run Matching Algorithm", variant="primary")

            match_status = gr.Textbox(label="Matching Algorithm Status", interactive=False)
            with gr.Row():
                match_table = gr.Dataframe(label="Ranked Compatibility Results")
                match_plot = gr.Image(label="Compatibility Score Graph")

            btn_match.click(run_organ_matching, inputs=[organ_select], outputs=[match_table, match_plot, match_status])

        # TAB 2: REGISTER DONOR
        with gr.TabItem("🎁 Register Organ Donor"):
            gr.Markdown("### Organ Donor Registration Form")
            with gr.Row():
                d_name = gr.Textbox(label="Donor Full Name", placeholder="e.g. David Miller")
                d_age = gr.Number(label="Age", value=30)
                d_gender = gr.Radio(choices=["Male", "Female", "Other"], label="Gender", value="Male")
            with gr.Row():
                d_blood = gr.Dropdown(choices=["O-", "O+", "A-", "A+", "B-", "B+", "AB-", "AB+"], label="Blood Group", value="O-")
                d_organ = gr.Dropdown(choices=["Kidney", "Liver", "Heart", "Lungs", "Pancreas", "Cornea"], label="Organ to Donate", value="Kidney")
            with gr.Row():
                d_loc = gr.Textbox(label="City / Location", placeholder="e.g. Bangalore")
                d_hosp = gr.Textbox(label="Hospital Name", placeholder="e.g. Apollo Hospital")
                d_contact = gr.Textbox(label="Contact Phone", placeholder="e.g. 9876543210")

            btn_reg_d = gr.Button("💾 Register Donor", variant="primary")
            d_output = gr.Textbox(label="Registration Status", interactive=False)
            btn_reg_d.click(register_donor, inputs=[d_name, d_age, d_gender, d_blood, d_organ, d_loc, d_hosp, d_contact], outputs=[d_output])

        # TAB 3: REGISTER RECIPIENT
        with gr.TabItem("🏥 Register Patient (Recipient)"):
            gr.Markdown("### Organ Recipient Registration Form")
            with gr.Row():
                r_name = gr.Textbox(label="Patient Full Name", placeholder="e.g. Alice Smith")
                r_age = gr.Number(label="Age", value=32)
                r_gender = gr.Radio(choices=["Male", "Female", "Other"], label="Gender", value="Female")
            with gr.Row():
                r_blood = gr.Dropdown(choices=["O-", "O+", "A-", "A+", "B-", "B+", "AB-", "AB+"], label="Patient Blood Group", value="A+")
                r_organ = gr.Dropdown(choices=["Kidney", "Liver", "Heart", "Lungs", "Pancreas", "Cornea"], label="Organ Needed", value="Kidney")
                r_urgency = gr.Dropdown(choices=["Critical (Level 1)", "High (Level 2)", "Medium (Level 3)", "Normal (Level 4)"], label="Medical Urgency Level", value="Critical (Level 1)")
            with gr.Row():
                r_loc = gr.Textbox(label="City / Location", placeholder="e.g. Bangalore")
                r_hosp = gr.Textbox(label="Hospital Name", placeholder="e.g. Fortis Hospital")
                r_contact = gr.Textbox(label="Contact Phone", placeholder="e.g. 9123456789")

            btn_reg_r = gr.Button("💾 Register Patient", variant="primary")
            r_output = gr.Textbox(label="Registration Status", interactive=False)
            btn_reg_r.click(register_recipient, inputs=[r_name, r_age, r_gender, r_blood, r_organ, r_urgency, r_loc, r_hosp, r_contact], outputs=[r_output])

        # TAB 4: DATABASE RECORDS
        with gr.TabItem("📊 Database Records"):
            btn_refresh_db = gr.Button("🔄 Refresh Database", variant="secondary")
            with gr.Row():
                db_donors = gr.Dataframe(label="Registered Donors Table")
                db_recipients = gr.Dataframe(label="Waiting Recipients Table")
            btn_refresh_db.click(view_database, outputs=[db_donors, db_recipients])

# Launch Web Application with Public Link
app.launch(share=True, debug=True)

ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)
ERROR: No matching distribution found for sqlite3


/tmp/ipykernel_1140/2412636836.py:170: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="Organ Donor & Receiver Website") as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://20ee510f5b053369e6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
